# Grid Up Datathon — 02 · Baseline

Amaç: **en hızlı geçerli submission**. Optimize etmeden önce çalışan bir uçtan
uca hattın olsun. İlk gün hedefi tek bir sayı: leaderboard'da bir skor.

Sıra: fold'lar → feature → eğit → doğrula → yaz.

In [ ]:
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from gridup import cross_validate, read_any, set_global_seed, write_submission
from gridup.compat import categorical_columns
from gridup.experiment import ExperimentLog, ExperimentRecord
from gridup.features import add_calendar_features, add_frequency_encoding
from gridup.metrics import inverse_log_transform, log_transform_target
from gridup.models import starter_params
from gridup.validation import build_splitter, purged_time_series_split

set_global_seed(42)

DATA_DIR = Path("/kaggle/input/GRID-UP-YARISMA-SLUG") if IS_KAGGLE else Path("../data/raw")
OUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("../submissions")

TARGET = "HEDEF_KOLON"     # TODO
ID_COLUMN = "id"           # TODO
TIME_COLUMN = None         # TODO
GROUP_COLUMN = None        # TODO
METRIC = "rmse"            # TODO — yarışmanın resmi metriği
TASK = "regression"        # regression | binary | multiclass
LOG_TARGET = False         # metrik RMSLE ise veya hedef çok çarpıksa True

In [ ]:
train = read_any(DATA_DIR / "train.csv")
test  = read_any(DATA_DIR / "test.csv")
print(train.shape, test.shape)

## 1 · Fold'lar — feature üretmeden ÖNCE

Sıra önemli: hedef kodlama fold'lara ihtiyaç duyar. Fold'ları önce sabitle ki
tüm deneyler **aynı bölmeler** üzerinde karşılaştırılabilir olsun.

In [ ]:
if TIME_COLUMN:
    train[TIME_COLUMN] = pd.to_datetime(train[TIME_COLUMN])
    test[TIME_COLUMN] = pd.to_datetime(test[TIME_COLUMN])
    # Ambargo: en uzun kayan pencerenden BÜYÜK olmalı
    folds = list(purged_time_series_split(train[TIME_COLUMN], n_splits=5,
                                          embargo=pd.Timedelta(days=30)))
elif GROUP_COLUMN:
    splitter = build_splitter("GroupKFold", n_splits=5)
    folds = list(splitter.split(train, groups=train[GROUP_COLUMN]))
else:
    scheme = "StratifiedKFold" if TASK != "regression" else "KFold"
    splitter = build_splitter(scheme, n_splits=5, seed=42)
    folds = list(splitter.split(train, train[TARGET] if TASK != "regression" else None))

for i, (tr, va) in enumerate(folds, 1):
    print(f"fold {i}: train={len(tr):>8,}  valid={len(va):>8,}")

## 2 · Feature'lar

**Kural:** train ve test'e *aynı* fonksiyon uygulanır. Ayrı kod yolları,
eğitim/servis uyumsuzluğunun bir numaralı kaynağıdır.

In [ ]:
def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Train ve test'e aynı dönüşümleri uygular. Girdiyi değiştirmez."""
    out = frame.copy()
    if TIME_COLUMN:
        out = add_calendar_features(out, TIME_COLUMN, include_year=False)
    # categorical_columns: pandas 2.x ve 3.x'te de doğru çalışır.
    # Düz `dtype == object` kontrolü pandas 3.0'da metin kolonlarını KAÇIRIR.
    categorical = categorical_columns(out)
    if categorical:
        out = add_frequency_encoding(out, categorical[:12])
    return out

train_features = build_features(train)
test_features = build_features(test)

drop = {TARGET, ID_COLUMN, TIME_COLUMN} - {None}
FEATURES = [c for c in train_features.columns
            if c not in drop and c in test_features.columns]
print(f"{len(FEATURES)} feature")

## 3 · Eğit

In [ ]:
y = train_features[TARGET].to_numpy()
if LOG_TARGET:
    y = log_transform_target(y)

params = starter_params("lightgbm", TASK)

result = cross_validate(
    train_features[FEATURES], y, folds,
    kind="lightgbm", task_type=TASK, metric=METRIC,
    params=params, test=test_features[FEATURES],
)

print(result.summary())

## 4 · Submission

`write_submission` yazmadan önce doğrular: NaN, sonsuz, eksik ID, sabit tahmin,
negatif değer. Kaggle'ın "Submission Scoring Error" mesajı sana hiçbir şey söylemez.

In [ ]:
predictions = result.test_predictions
if LOG_TARGET:
    predictions = inverse_log_transform(predictions)

path = write_submission(
    test_features[ID_COLUMN].to_numpy(),
    predictions,
    OUT_DIR / "baseline_lgbm.csv",
    id_column=ID_COLUMN,
    target_column=TARGET,
)

## 5 · Deney defterine yaz

Submission gönderdikten **sonra** leaderboard skorunu geri yaz:

```python
log.record_lb("baseline_lgbm", 12.3456)
print(log.cv_lb_correlation())
```

CV–LB korelasyonu bu yarışmanın en önemli tek sayısıdır. r > 0.8 ise CV'ne
güven; r < 0.5 ise CV şemanı düzeltmeden devam etme.

In [ ]:
log = ExperimentLog(OUT_DIR.parent / "experiments" / "deneyler.jsonl")

log.add(ExperimentRecord(
    name="baseline_lgbm",
    cv_score=result.overall_score,
    metric=METRIC,
    model_kind="lightgbm",
    n_features=len(FEATURES),
    fold_scores=result.fold_scores,
    notes="baseline: takvim + frekans kodlama",
    submission_path=str(path),
))

log.leaderboard()

## Sonraki adımlar

1. **Adversarial validation** — `validation.adversarial_validation(train, test)`
   ile train/test kayması var mı ölç
2. **Hedef kodlama** — `features.oof_target_encode` (yüksek kardinaliteli kolonlar)
3. **Lag / rolling** — zaman varsa en güçlü feature ailesi
4. **Harici veri** — `scripts/fetch_weather.py` ile hava durumu
5. **CatBoost + XGBoost** — çeşitlilik için, sonra `ensemble.hill_climb_weights`
6. **Eşik optimizasyonu** — sınıflandırmaysa `metrics.optimize_threshold`